# Customer Churn Prediction Using Machine Learning

## Project Overview

Customer churn refers to customers who stop using a company's service. Predicting customer churn helps companies identify customers who are likely to leave and take preventive actions.

In this project, we use Machine Learning to predict whether a telecom customer will churn based on customer information such as contract type, monthly charges, tenure, internet service, payment method, and other features.

## Objectives

- Understand the dataset
- Perform Data Cleaning
- Perform Exploratory Data Analysis (EDA)
- Build Machine Learning Models
- Compare different models
- Predict Customer Churn

In [ ]:
# ===============================
# Customer Churn Prediction
# Import Required Libraries
# ===============================

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline

print("Libraries Imported Successfully")

In [ ]:
url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"

df = pd.read_csv(url)

print("Dataset Loaded Successfully")

In [ ]:
df.head()

In [ ]:
import pandas as pd
url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
df = pd.read_csv(url)
df.shape

In [ ]:
df.columns

In [ ]:
df.info()

In [ ]:
df.head(10)

In [ ]:
df.tail(10)

In [ ]:
print("Rows and Columns:", df.shape)

In [ ]:
print(df.columns.tolist())

In [ ]:
df.dtypes

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.describe(include="object")

In [ ]:
df.isnull().sum()

In [ ]:
print("Duplicate Rows:", df.duplicated().sum())

In [ ]:
df["Churn"].value_counts()

In [ ]:
df["Churn"].value_counts(normalize=True)*100

In [ ]:
df.nunique()

In [ ]:
df.sample(5)

## Dataset Exploration Summary

The IBM Telco Customer Churn dataset contains customer demographic information, account details, subscribed services, billing information, and churn status.

During the initial exploration, the dataset structure, column names, data types, missing values, duplicate records, and target variable distribution were examined. The dataset consists of 7,043 customer records and 21 attributes. Most features are categorical, while tenure, MonthlyCharges, and TotalCharges represent numerical information.

The target variable (Churn) indicates whether a customer left the company. Approximately three-quarters of customers remained with the company, while about one-quarter churned.

This exploration provides a better understanding of the dataset before proceeding with data cleaning and visualization.

In [ ]:
df["TotalCharges"].head(10)

In [ ]:
print(df["TotalCharges"].dtype)

In [ ]:
blank_total_charges = (df["TotalCharges"].str.strip() == "").sum()

print("Blank values in TotalCharges:", blank_total_charges)

In [ ]:
df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"],
    errors="coerce"
)

In [ ]:
print(df["TotalCharges"].dtype)

In [ ]:
df.isnull().sum()

In [ ]:
df[df["TotalCharges"].isnull()]

In [ ]:
df = df.dropna().copy()

In [ ]:
df.reset_index(drop=True, inplace=True)

In [ ]:
print("Duplicates before removal:", df.duplicated().sum())

In [ ]:
df = df.drop_duplicates().copy()

In [ ]:
print("Total rows:", len(df))
print("Unique customer IDs:", df["customerID"].nunique())

In [ ]:
print("Duplicate customer IDs:", df["customerID"].duplicated().sum())

In [ ]:
df[["tenure", "MonthlyCharges", "TotalCharges"]].describe()

In [ ]:
print("Negative tenure values:", (df["tenure"] < 0).sum())
print("Negative monthly charges:", (df["MonthlyCharges"] < 0).sum())
print("Negative total charges:", (df["TotalCharges"] < 0).sum())

In [ ]:
categorical_columns = df.select_dtypes(include="object").columns

for column in categorical_columns:
    print(f"\n{column}")
    print(df[column].value_counts())

In [ ]:
print("Cleaned Dataset Shape:", df.shape)
print("Total Missing Values:", df.isnull().sum().sum())
print("Duplicate Rows:", df.duplicated().sum())
print("\nData Types:")
print(df.dtypes)

In [ ]:
df.to_csv(
    "Telco_Customer_Churn_Cleaned.csv",
    index=False
)

print("Cleaned dataset saved successfully.")

In [ ]:
from google.colab import files

files.download("Telco_Customer_Churn_Cleaned.csv")

## Data Cleaning

The `TotalCharges` column was initially stored as an object data type because it contained blank values. It was converted into a numerical data type using `pd.to_numeric()` with invalid entries replaced by missing values.

Rows containing missing `TotalCharges` values were removed because they represented only a very small proportion of the complete dataset. Duplicate records were also checked and removed where necessary.

After cleaning, the dataset contained 7,032 customer records and 21 columns. The cleaned dataset had no missing values or duplicate records and was ready for Exploratory Data Analysis and machine learning.

In [ ]:
import os

os.makedirs("images", exist_ok=True)

print("Images folder created successfully.")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(6,5))

ax = sns.countplot(data=df, x="Churn", palette="Set2")

plt.title("Customer Churn Distribution", fontsize=14)
plt.xlabel("Churn")
plt.ylabel("Number of Customers")

for container in ax.containers:
    ax.bar_label(container)

plt.tight_layout()

plt.savefig("images/churn_distribution.png", dpi=300)

plt.show()

### Observation

The majority of customers did not churn, while a smaller proportion left the company. This indicates that the dataset is moderately imbalanced, which should be considered during model evaluation.

In [ ]:
plt.figure(figsize=(7,5))

sns.countplot(
    data=df,
    x="gender",
    hue="Churn",
    palette="Set1"
)

plt.title("Customer Churn by Gender")
plt.xlabel("Gender")
plt.ylabel("Customers")

plt.tight_layout()

plt.savefig("images/gender_churn.png", dpi=300)

plt.show()

### Observation

Customer churn appears to be similar for both male and female customers. Gender alone does not seem to significantly influence customer churn.

In [ ]:
plt.figure(figsize=(8,5))

sns.countplot(
    data=df,
    x="Contract",
    hue="Churn",
    palette="viridis"
)

plt.title("Customer Churn by Contract Type")

plt.xticks(rotation=10)

plt.tight_layout()

plt.savefig("images/contract_churn.png", dpi=300)

plt.show()

### Observation

Customers with month-to-month contracts exhibit the highest churn rate. Customers with one-year and two-year contracts are considerably less likely to churn.

In [ ]:
plt.figure(figsize=(8,5))

sns.countplot(
    data=df,
    x="InternetService",
    hue="Churn",
    palette="coolwarm"
)

plt.title("Customer Churn by Internet Service")

plt.tight_layout()

plt.savefig("images/internet_service.png", dpi=300)

plt.show()

### Observation

Customers using fiber optic internet appear to churn more frequently than those using DSL or having no internet service.

In [ ]:
plt.figure(figsize=(11,5))

sns.countplot(
    data=df,
    x="PaymentMethod",
    hue="Churn"
)

plt.xticks(rotation=25)

plt.title("Customer Churn by Payment Method")

plt.tight_layout()

plt.savefig("images/payment_method.png", dpi=300)

plt.show()

### Observation

Customers who use electronic checks tend to have higher churn compared to customers using automatic payment methods.

In [ ]:
plt.figure(figsize=(7,5))

sns.boxplot(
    data=df,
    x="Churn",
    y="MonthlyCharges",
    palette="Set3"
)

plt.title("Monthly Charges by Churn")

plt.tight_layout()

plt.savefig("images/monthly_charges.png", dpi=300)

plt.show()

### Observation

Customers who churn generally have higher monthly charges than customers who remain with the company.

In [ ]:
plt.figure(figsize=(8,5))

sns.histplot(
    data=df,
    x="tenure",
    hue="Churn",
    bins=30,
    multiple="stack"
)

plt.title("Customer Tenure Distribution")

plt.tight_layout()

plt.savefig("images/tenure_distribution.png", dpi=300)

plt.show()

### Observation

Most customers who churn have relatively short tenure. Long-term customers are more likely to remain with the company.

In [ ]:
numeric_df = df.select_dtypes(include=["int64","float64"])

In [ ]:
plt.figure(figsize=(8,6))

sns.heatmap(
    numeric_df.corr(),
    annot=True,
    cmap="coolwarm",
    linewidths=0.5
)

plt.title("Correlation Heatmap")

plt.tight_layout()

plt.savefig("images/correlation_heatmap.png", dpi=300)

plt.show()

### Observation

The correlation matrix shows that tenure and TotalCharges have a strong positive relationship. MonthlyCharges has a moderate relationship with TotalCharges, while other variables show relatively weak correlations.

In [ ]:
plt.figure(figsize=(6,6))

df["Churn"].value_counts().plot(
    kind="pie",
    autopct="%1.1f%%",
    startangle=90
)

plt.ylabel("")

plt.title("Customer Churn Percentage")

plt.tight_layout()

plt.savefig("images/churn_pie.png", dpi=300)

plt.show()

# Exploratory Data Analysis Summary

The exploratory data analysis identified several important customer churn patterns.

Customers with month-to-month contracts showed the highest churn rates. Fiber optic internet users also appeared more likely to churn compared to DSL users. Customers paying through electronic checks demonstrated relatively higher churn behaviour.

The analysis also revealed that customers with shorter tenure and higher monthly charges were more likely to discontinue the service. Correlation analysis indicated a strong relationship between customer tenure and total charges.

These insights provide valuable information for feature engineering and machine learning model development.

In [ ]:
df["Churn"] = df["Churn"].map({
    "No": 0,
    "Yes": 1
})

In [ ]:
df["Churn"].value_counts()

0 = Customer Stayed

1 = Customer Left

In [ ]:
df = df.drop("customerID", axis=1)

In [ ]:
df.head()

In [ ]:
X = df.drop("Churn", axis=1)

y = df["Churn"]

In [ ]:
print(X.shape)

print(y.shape)

X → Input Features

y → Output(Target)

In [ ]:
categorical_columns = X.select_dtypes(
    include="object"
).columns

print(categorical_columns)

In [ ]:
X = pd.get_dummies(
    X,
    drop_first=True
)

In [ ]:
X.head()

In [ ]:
print(X.shape)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(

    X,

    y,

    test_size=0.2,

    random_state=42,

    stratify=y

)

In [ ]:
print(X_train.shape)

print(X_test.shape)

In [ ]:
from sklearn.preprocessing import StandardScaler

In [ ]:
scaler = StandardScaler()

In [ ]:
X_train_scaled = scaler.fit_transform(
    X_train
)

In [ ]:
X_test_scaled = scaler.transform(
    X_test
)

In [ ]:
print(X_train_scaled.shape)

In [ ]:
import joblib

joblib.dump(
    scaler,
    "scaler.pkl"
)

# Data Preprocessing Summary

The target variable was converted into binary numerical values, where 0 represents customers who remained with the company and 1 represents customers who churned.

The customer ID column was removed because it does not contribute to predicting customer churn.

Categorical variables were transformed into numerical features using one-hot encoding. The dataset was then divided into training and testing sets using an 80:20 ratio while preserving the original churn distribution through stratified sampling.

Finally, feature scaling was performed using StandardScaler to normalize the numerical feature values before training machine learning models.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

In [ ]:
logistic_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

In [ ]:
logistic_model.fit(
    X_train_scaled,
    y_train
)

In [ ]:
logistic_predictions = logistic_model.predict(
    X_test_scaled
)

In [ ]:
logistic_probabilities = logistic_model.predict_proba(
    X_test_scaled
)[:,1]

In [ ]:
rf_model = RandomForestClassifier(

    n_estimators=300,

    random_state=42,

    class_weight="balanced"
)

In [ ]:
rf_model.fit(

    X_train,

    y_train

)

In [ ]:
rf_predictions = rf_model.predict(
    X_test
)

In [ ]:
rf_probabilities = rf_model.predict_proba(
    X_test
)[:,1]

In [ ]:
print("Logistic Predictions")

print(logistic_predictions[:20])

print()

print("Random Forest Predictions")

print(rf_predictions[:20])

In [ ]:
print(len(logistic_predictions))

print(len(rf_predictions))

print(len(y_test))

In [ ]:
import joblib

joblib.dump(
    logistic_model,
    "logistic_regression.pkl"
)

joblib.dump(
    rf_model,
    "random_forest.pkl"
)

print("Models Saved Successfully")

# Machine Learning Model Development

Two supervised machine learning algorithms were developed for customer churn prediction.

## Logistic Regression

Logistic Regression was selected as a baseline classification model because of its simplicity, efficiency, and interpretability. The model was trained using standardized numerical features.

## Random Forest

Random Forest was selected because it combines multiple decision trees to improve prediction performance and reduce overfitting. Unlike Logistic Regression, Random Forest does not require feature scaling.

Both models were trained using the prepared training dataset and were subsequently evaluated using multiple performance metrics.

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    RocCurveDisplay
)

In [ ]:
logistic_accuracy = accuracy_score(y_test, logistic_predictions)
logistic_precision = precision_score(y_test, logistic_predictions)
logistic_recall = recall_score(y_test, logistic_predictions)
logistic_f1 = f1_score(y_test, logistic_predictions)
logistic_auc = roc_auc_score(y_test, logistic_probabilities)

print("===== Logistic Regression =====")
print(f"Accuracy : {logistic_accuracy:.4f}")
print(f"Precision: {logistic_precision:.4f}")
print(f"Recall   : {logistic_recall:.4f}")
print(f"F1 Score : {logistic_f1:.4f}")
print(f"ROC AUC  : {logistic_auc:.4f}")

In [ ]:
print(classification_report(
    y_test,
    logistic_predictions,
    target_names=["Stayed", "Churned"]
))

In [ ]:
rf_accuracy = accuracy_score(y_test, rf_predictions)
rf_precision = precision_score(y_test, rf_predictions)
rf_recall = recall_score(y_test, rf_predictions)
rf_f1 = f1_score(y_test, rf_predictions)
rf_auc = roc_auc_score(y_test, rf_probabilities)

print("===== Random Forest =====")
print(f"Accuracy : {rf_accuracy:.4f}")
print(f"Precision: {rf_precision:.4f}")
print(f"Recall   : {rf_recall:.4f}")
print(f"F1 Score : {rf_f1:.4f}")
print(f"ROC AUC  : {rf_auc:.4f}")

In [ ]:
print(classification_report(
    y_test,
    rf_predictions,
    target_names=["Stayed", "Churned"]
))

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_test,
    logistic_predictions,
    cmap="Blues",
    display_labels=["Stayed","Churned"]
)

plt.title("Logistic Regression Confusion Matrix")
plt.show()

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_test,
    rf_predictions,
    cmap="Greens",
    display_labels=["Stayed","Churned"]
)

plt.title("Random Forest Confusion Matrix")
plt.show()

In [ ]:
plt.figure(figsize=(8,6))

RocCurveDisplay.from_predictions(
    y_test,
    logistic_probabilities,
    name="Logistic Regression"
)

RocCurveDisplay.from_predictions(
    y_test,
    rf_probabilities,
    name="Random Forest"
)

plt.title("ROC Curve Comparison")

plt.show()

In [ ]:
comparison = pd.DataFrame({

    "Model":[
        "Logistic Regression",
        "Random Forest"
    ],

    "Accuracy":[
        logistic_accuracy,
        rf_accuracy
    ],

    "Precision":[
        logistic_precision,
        rf_precision
    ],

    "Recall":[
        logistic_recall,
        rf_recall
    ],

    "F1 Score":[
        logistic_f1,
        rf_f1
    ],

    "ROC AUC":[
        logistic_auc,
        rf_auc
    ]

})

comparison.round(4)

In [ ]:
comparison.set_index("Model").plot(

    kind="bar",

    figsize=(10,6)

)

plt.title("Machine Learning Model Comparison")

plt.ylabel("Score")

plt.ylim(0,1)

plt.xticks(rotation=0)

plt.grid(axis="y", alpha=0.3)

plt.show()

In [ ]:
if rf_f1 > logistic_f1:

    best_model = "Random Forest"

else:

    best_model = "Logistic Regression"

print("Best Model:", best_model)

In [ ]:
if best_model == "Random Forest":

    joblib.dump(rf_model, "best_model.pkl")

else:

    joblib.dump(logistic_model, "best_model.pkl")

print("Best model saved successfully.")

# Model Evaluation Summary

Two machine learning models were evaluated for customer churn prediction: Logistic Regression and Random Forest.

Both models were assessed using Accuracy, Precision, Recall, F1 Score, ROC-AUC Score, Classification Reports, Confusion Matrices, and ROC Curves.

The comparison showed that **(replace this after running the notebook with the actual best model)** achieved the best overall performance based on the F1 Score and ROC-AUC. Therefore, it was selected as the final prediction model for this project.

In [ ]:
plt.savefig("images/model_comparison.png", dpi=300)

In [ ]:
feature_importance = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": rf_model.feature_importances_
})

feature_importance = feature_importance.sort_values(
    by="Importance",
    ascending=False
)

feature_importance.head(15)

In [ ]:
top_features = feature_importance.head(15)

plt.figure(figsize=(10, 7))

sns.barplot(
    data=top_features,
    x="Importance",
    y="Feature"
)

plt.title("Top 15 Features Influencing Customer Churn")
plt.xlabel("Feature Importance")
plt.ylabel("Feature")

plt.tight_layout()

plt.savefig(
    "images/feature_importance.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
logistic_coefficients = pd.DataFrame({
    "Feature": X_train.columns,
    "Coefficient": logistic_model.coef_[0]
})

logistic_coefficients["Absolute_Coefficient"] = (
    logistic_coefficients["Coefficient"].abs()
)

logistic_coefficients = logistic_coefficients.sort_values(
    by="Absolute_Coefficient",
    ascending=False
)

logistic_coefficients.head(15)

In [ ]:
positive_features = logistic_coefficients.sort_values(
    by="Coefficient",
    ascending=False
).head(10)

positive_features[
    ["Feature", "Coefficient"]
]

In [ ]:
negative_features = logistic_coefficients.sort_values(
    by="Coefficient",
    ascending=True
).head(10)

negative_features[
    ["Feature", "Coefficient"]
]

In [ ]:
top_logistic_features = logistic_coefficients.head(15).sort_values(
    by="Coefficient"
)

plt.figure(figsize=(10, 7))

sns.barplot(
    data=top_logistic_features,
    x="Coefficient",
    y="Feature"
)

plt.title("Top Logistic Regression Coefficients")
plt.xlabel("Coefficient")
plt.ylabel("Feature")

plt.tight_layout()

plt.savefig(
    "images/logistic_coefficients.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# Feature Importance and Business Insights

The feature importance analysis was used to identify the customer characteristics that contributed most strongly to churn predictions.

The results indicate that factors such as customer tenure, monthly charges, total charges, contract type, internet service type, payment method, online security, and technical support can play an important role in predicting churn.

Customers with shorter tenure, month-to-month contracts, higher monthly charges, electronic check payments, and limited support services may be more likely to leave the company.

Based on these findings, the company could consider the following retention strategies:

- Provide special onboarding support for newly joined customers
- Encourage customers to move from month-to-month contracts to long-term contracts
- Review high monthly charges and offer suitable loyalty discounts
- Promote automatic payment methods
- Offer online security and technical support packages
- Contact high-risk customers before they decide to leave

Feature importance identifies predictive relationships, but it does not prove that one feature directly causes churn.

In [ ]:
if best_model == "Random Forest":
    churn_probabilities = rf_model.predict_proba(X_test)[:, 1]
else:
    churn_probabilities = logistic_model.predict_proba(
        X_test_scaled
    )[:, 1]

In [ ]:
risk_results = X_test.copy()

risk_results["Actual_Churn"] = y_test.values
risk_results["Churn_Probability"] = churn_probabilities
risk_results["Predicted_Churn"] = (
    churn_probabilities >= 0.5
).astype(int)

In [ ]:
high_risk_customers = risk_results.sort_values(
    by="Churn_Probability",
    ascending=False
).head(20)

high_risk_customers[
    [
        "Actual_Churn",
        "Predicted_Churn",
        "Churn_Probability"
    ]
]

In [ ]:
def assign_risk_level(probability):
    if probability >= 0.75:
        return "High Risk"
    elif probability >= 0.50:
        return "Medium Risk"
    else:
        return "Low Risk"


risk_results["Risk_Level"] = risk_results[
    "Churn_Probability"
].apply(assign_risk_level)

In [ ]:
risk_results["Risk_Level"].value_counts()

In [ ]:
plt.figure(figsize=(7, 5))

ax = sns.countplot(
    data=risk_results,
    x="Risk_Level",
    order=["Low Risk", "Medium Risk", "High Risk"]
)

plt.title("Customer Churn Risk Levels")
plt.xlabel("Risk Category")
plt.ylabel("Number of Customers")

for container in ax.containers:
    ax.bar_label(container)

plt.tight_layout()

plt.savefig(
    "images/churn_risk_levels.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
risk_results.to_csv(
    "customer_churn_risk_predictions.csv",
    index=False
)

print("Prediction results saved successfully.")

In [ ]:
import os
import shutil

folders = [
    "data",
    "models",
    "notebook",
    "results",
    "images"
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)

print("Project folders created successfully.")

In [ ]:
files_to_move = {
    "Telco-Customer-Churn.csv": "data/Telco-Customer-Churn.csv",
    "Telco_Customer_Churn_Cleaned.csv": "data/Telco_Customer_Churn_Cleaned.csv",
    "logistic_regression.pkl": "models/logistic_regression.pkl",
    "random_forest.pkl": "models/random_forest.pkl",
    "best_model.pkl": "models/best_model.pkl",
    "scaler.pkl": "models/scaler.pkl",
    "customer_churn_risk_predictions.csv": "results/customer_churn_risk_predictions.csv"
}

for source, destination in files_to_move.items():
    if os.path.exists(source):
        shutil.move(source, destination)
        print(f"Moved: {source} → {destination}")
    else:
        print(f"Not found: {source}")

In [ ]:
requirements_content = """
pandas
numpy
matplotlib
seaborn
scikit-learn
joblib
"""

with open("requirements.txt", "w") as file:
    file.write(requirements_content.strip())

print("requirements.txt created successfully.")

In [ ]:
gitignore_content = """
__pycache__/
*.pyc
.ipynb_checkpoints/
.env
.DS_Store
"""

with open(".gitignore", "w") as file:
    file.write(gitignore_content.strip())

print(".gitignore created successfully.")

In [ ]:
license_content = """
MIT License

Copyright (c) 2026 sithmi umeshika

Permission is hereby granted, free of charge, to any person obtaining a copy
of this software and associated documentation files, to deal in the Software
without restriction, including without limitation the rights to use, copy,
modify, merge, publish, distribute, sublicense, and/or sell copies of the
Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND.
"""

with open("LICENSE", "w") as file:
    file.write(license_content.strip())

print("LICENSE created successfully.")

In [ ]:
shutil.make_archive(
    "customer-churn-prediction",
    "zip",
    "."
)

print("Project ZIP created successfully.")

In [ ]:
from google.colab import files

files.download("customer-churn-prediction.zip")

In [ ]:
comparison.round(4)

In [ ]:
print("Selected Best Model:", best_model)

In [ ]:
readme_content = f"""
# Customer Churn Prediction Using Machine Learning

![Python](https://img.shields.io/badge/Python-3.x-blue)
![Machine Learning](https://img.shields.io/badge/Machine%20Learning-Classification-orange)
![Scikit-learn](https://img.shields.io/badge/Scikit--learn-Modeling-yellow)
![Status](https://img.shields.io/badge/Project-Completed-success)
![License](https://img.shields.io/badge/License-MIT-green)

## Project Overview

Customer churn occurs when an existing customer stops using a company's products or services. Predicting churn allows companies to identify customers who are likely to leave and take suitable retention actions.

This project develops an end-to-end machine learning solution to predict customer churn in a telecommunications company. It includes data cleaning, exploratory data analysis, feature engineering, model development, performance evaluation, customer risk classification, and business recommendations.

---

## Business Problem

Customer acquisition is often more expensive than customer retention. Therefore, telecom companies need an effective way to identify customers who are at risk of leaving.

The main objective of this project is to answer the following question:

> Can machine learning identify customers who are likely to churn using their service, contract, billing, and account information?

---

## Project Objectives

- Understand and clean the telecom customer dataset
- Identify customer churn patterns through exploratory data analysis
- Transform categorical data into machine-readable features
- Develop Logistic Regression and Random Forest models
- Evaluate models using multiple classification metrics
- Identify the most important churn-related features
- Classify customers into low, medium, and high-risk groups
- Provide practical customer-retention recommendations

---

## Dataset

The project uses the IBM Telco Customer Churn dataset.

The dataset originally contained:

- **7,043 customer records**
- **21 columns**
- Customer demographic information
- Subscribed service information
- Contract and payment details
- Monthly and total charges
- Customer churn status

After data cleaning, the final dataset contained **7,032 records**.

### Target Variable

- `0` – Customer remained with the company
- `1` – Customer churned

---

## Technologies Used

- Python
- Pandas
- NumPy
- Matplotlib
- Seaborn
- Scikit-learn
- Joblib
- Google Colab
- GitHub

---

## Project Workflow

```text
Business Understanding
        ↓
Data Collection
        ↓
Data Exploration
        ↓
Data Cleaning
        ↓
Exploratory Data Analysis
        ↓
Feature Engineering
        ↓
Train-Test Split
        ↓
Model Development
        ↓
Model Evaluation
        ↓
Feature Importance
        ↓
Customer Risk Classification
        ↓
Business Recommendations
"""

with open("README.md", "w") as file:
    file.write(readme_content.strip())

print("README.md created successfully.")

In [ ]:
import os

print(os.path.exists("README.md"))

In [ ]:
from google.colab import files

files.download("README.md")

In [ ]:
with open("README.md", "r", encoding="utf-8") as file:
    print(file.read()[:2000])

In [ ]:
github_username = "sithmi_umeshika"

with open("README.md", "r", encoding="utf-8") as file:
    content = file.read()

content = content.replace(
    "YOUR-USERNAME",
    github_username
)

with open("README.md", "w", encoding="utf-8") as file:
    file.write(content)

print("GitHub username updated successfully.")

In [ ]:
import os

print("README exists:", os.path.exists("README.md"))

In [ ]:
import shutil

shutil.make_archive(
    "customer-churn-prediction-final",
    "zip",
    "."
)

print("Final ZIP created successfully.")

In [ ]:
from google.colab import files

files.download("customer-churn-prediction-final.zip")

In [ ]:
import os

required_images = [
    "churn_distribution.png",
    "contract_churn.png",
    "monthly_charges.png",
    "tenure_distribution.png",
    "payment_method.png",
    "model_comparison.png",
    "roc_curve.png",
    "feature_importance.png",
    "churn_risk_levels.png"
]

for image in required_images:
    path = os.path.join("images", image)

    if os.path.exists(path):
        print("Found:", path)
    else:
        print("Missing:", path)

In [ ]:
for root, directories, files in os.walk("."):
    level = root.replace(".", "").count(os.sep)
    indent = "    " * level

    print(f"{indent}{os.path.basename(root)}/")

    for filename in files:
        print(f"{indent}    {filename}")

In [ ]:
import shutil

shutil.make_archive(
    "customer-churn-prediction-final",
    "zip",
    "."
)

print("Final project ZIP created successfully.")